# Data Preprocessing Pipeline

This notebook prepares the Telco Customer Churn dataset for machine learning by applying:
- leakage-safe preprocessing
- feature engineering
- categorical encoding
- feature scaling

“Not all high-risk customers should be targeted. Retention efforts should focus on customers who are both high-risk and high-value to maximize ROI.”

In [1]:
# load data

import pandas as pd

df = pd.read_csv('../data/raw_data/rawdata-Telco-Customer-Churn.csv')

In [2]:
# Data Cleaning

df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = df.dropna()

In [3]:
# We do not need customer ID in this analysis... so drop it!

df = df.drop(columns=['customerID'])

In [4]:
# Target Encoding... 0,1 label required

df['Churn'] = df['Churn'].map({'Yes':1, 'No':0})

### Train/Test Split

**Preventing Data Leakage**

To ensure realistic model evaluation, train-test split is performed before train-dependent transformations such as scaling and threshold-based feature creation.

**stratify=y**

Important for imbalanced data – keeps the same ratio of "Churn=Yes" / "Churn=No" in both train and test sets. Without this, one set might accidentally have very few churn examples.



In [5]:
from sklearn.model_selection import train_test_split

X = df.drop('Churn', axis=1)
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

## Feature Engineering Function

A reusable feature engineering pipeline is created to ensure consistent transformations across training and test datasets while preventing data leakage.

In [ ]:
def feature_engineering(df):
    
    df = df.copy()

    # =========================
    # 1. Avg Charges
    # =========================
    
    df['AvgCharges'] = (
        df['TotalCharges'] / (df['tenure'] + 1)
    )

    
    # =========================
    # 2. ChargeDifference
    # =========================
    
    df['ChargeDifference'] = (
        df['MonthlyCharges'] - df['AvgCharges']
    )

    
    # =========================
    # 3. Service Coverage Ratio
    # =========================
    
    services = [
        'OnlineSecurity',
        'OnlineBackup',
        'DeviceProtection',
        'TechSupport',
        'StreamingTV',
        'StreamingMovies'
    ]

    service_mapping = {
        'Yes': 1,
        'No': 0,
        'No internet service': 0
    }

    for col in services:
        df[col] = df[col].map(service_mapping)

    df['ServiceCoverageRatio'] = (
        df[services].sum(axis=1) / len(services)
    )

    
    # =========================
    # 4. Customer Value
    # =========================
    
    df['CustomerValue'] = (
        df['MonthlyCharges'] * df['tenure']
    )

    
    # =========================
    # 5. Contract Risk
    # =========================
    
    contract_mapping = {
        'Month-to-month': 2,
        'One year': 1,
        'Two year': 0
    }

    df['ContractRisk'] = df['Contract'].map(contract_mapping)
    
    # Column contract is not required anymore... drop it then
    df = df.drop(columns=['Contract'])

    
    # =========================
    # 6. Early Lifecycle Risk
    # =========================
    
    df['EarlyLifecycle'] = (
        df['tenure'] < 12
    ).astype(int)

    
    # =========================
    # 7. Support Gap
    # =========================
    
    df['SupportGap'] = (
        (df['TechSupport'] == 0) &
        (df['OnlineSecurity'] == 0)
    ).astype(int)

    
    return df

In [7]:
# Apply feature engineering separately

X_train = feature_engineering(X_train)
X_test = feature_engineering(X_test)

## Binary Feature Encoding

Binary categorical variables are encoded into 0/1 representations to simplify model learning while preserving semantic meaning.

In [8]:
binary_cols = ['gender','Partner','Dependents','PhoneService','PaperlessBilling']

for col in binary_cols:
    X_train[col] = X_train[col].map({
        'Yes':1,
        'No':0,
        'Female':1,
        'Male':0
    })

    X_test[col] = X_test[col].map({
        'Yes':1,
        'No':0,
        'Female':1,
        'Male':0
    })

In [9]:
X_train = pd.get_dummies(X_train, drop_first=True)
X_test = pd.get_dummies(X_test, drop_first=True)

# Align columns
X_train, X_test = X_train.align(
    X_test,
    join='left',
    axis=1,
    fill_value=0
)

## Feature Scaling

We scale **only continuous numerical features** to have zero mean and unit variance.  
This helps gradient-based models (e.g., Logistic Regression, SVM) converge faster.

**Columns scaled:** tenure, MonthlyCharges, TotalCharges, CustomerValue, AvgCharges, ChargeDifference

**Note:** Binary/dummy features (0/1) are **not** scaled to keep interpretability.

In [10]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

num_cols = [
    'tenure',
    'MonthlyCharges',
    'TotalCharges',
    'CustomerValue',
    'AvgCharges',
    'ChargeDifference'
]

scaler.fit(X_train[num_cols])

X_train[num_cols] = scaler.transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

In [11]:
# Save Processed Data

X_train.to_csv('../data/processed/X_train.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv', index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)

## Saving the Scaler

The `StandardScaler` was fitted on the training data to compute mean and standard deviation of numerical features.
To apply the **same transformation** to future data (during inference), we must save the scaler object.

**Why?**
- New data comes with its own scale.
- Retraining the scaler on new data would change the transformation, breaking model compatibility.  
- Saving ensures consistent preprocessing in production.

In [12]:
import joblib
joblib.dump(scaler, '../models/scaler.pkl')

['../models/scaler.pkl']